# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Skill loaded:** `building-baselines/SKILL.md` (+ `flyrank/flyrank-data/SKILL.md` for the dataset gotchas), per the card's own routing table.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Before I trust a rule, I check the signals it leans on. Two signals below, one bucket table each, with `n` printed, and an honest verdict.


In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print(f"rows={len(df):,}  cols={df.shape[1]}")
df[["impressions_90d","ctr","avg_position","days_since_last_update","freshness_tier","position_tier","trend_direction"]].head(3)


rows=30,000  cols=44


,impressions_90d,ctr,avg_position,days_since_last_update,freshness_tier,position_tier,trend_direction
0,3803,0.76,10.6,20,0-30,striking,down
1,15320,0.05,20.3,25,0-30,page_3_5,down
2,12581,0.09,36.5,20,0-30,page_3_5,down


### Signal 1 — staleness (the signal behind FlyRank's *refresh* flags)

**Claim:** pages that haven't been updated in a long time are more likely to be declining, so `days_since_last_update` is a fair basis for a "needs refresh" flag.

**Test:** among *visible* pages (`impressions_90d >= 100`, so I'm not testing noise), bucket by `freshness_tier` and compare the rate of `trend_direction == "down"`. I collapse `91-180` and `181+` into one `91+` bucket first — the raw `181+` bucket only has 35 rows, under the ~50-row floor, and would produce a noisy verdict on its own.


In [2]:
visible = df[df["impressions_90d"] >= 100].copy()
visible["is_declining"] = (visible["trend_direction"] == "down").astype(int)
visible["fresh_bucket"] = visible["freshness_tier"].replace({"181+": "91+", "91-180": "91+"})

signal1 = (
    visible.groupby("fresh_bucket")
    .agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean"))
    .reindex(["0-30", "31-90", "91+"])
    .round(3)
)
base_rate = visible["is_declining"].mean()
print(f"overall base decline rate (n={len(visible):,}): {base_rate:.3f}\n")
print(signal1)


overall base decline rate (n=22,006): 0.598

                  n  decline_rate
fresh_bucket                     
0-30          13735         0.583
31-90           152         0.592
91+            8119         0.623


**Verdict: MIXED**

Direction is right — decline rate rises from 0.583 (freshest) to 0.623 (91+ days stale) — but the effect is small, only about 4 points above the 0.598 base rate, and the middle bucket (`31-90`, n=152) barely moves off the freshest bucket at all. Staleness alone is a weak signal in this slice. It's real, but it's not strong enough to gate a rule by itself — I'll use it as a light secondary factor, not the main driver.


### Signal 2 — CTR vs. position (the signal behind FlyRank's *CTR-fix* logic)

**Claim:** click-through rate should track ranking position — better position, higher expected CTR — so a page with a *good* position but *low* CTR relative to its own position tier is a meaningful anomaly worth reviewing, not noise.

**Test:** among pages with real position data (`avg_position > 0`) and real visibility (`impressions_90d >= 100`), bucket by `position_tier` and compare median CTR.


In [3]:
posvis = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)].copy()

signal2 = (
    posvis.groupby("position_tier")
    .agg(n=("ctr", "size"), median_ctr=("ctr", "median"), mean_ctr=("ctr", "mean"))
    .reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
    .round(3)
)
print(signal2)


                  n  median_ctr  mean_ctr
position_tier                            
top_3           533        0.19     0.334
page_1         8633        0.23     0.355
striking       5903        0.15     0.256
page_3_5       6058        0.06     0.142
deep            879        0.00     0.055


**Verdict: CONFIRMED**

Median CTR falls in a clean, monotonic staircase from 0.19% (`top_3`) down to 0.00% (`deep`), every bucket well above the n-floor (smallest is 533). Position really does set the expectation for CTR here. That means "CTR far below the median for this page's own tier" is a meaningfully different situation from "just low CTR overall" — it's the right way to spot underperformers instead of penalizing every low-position page for having low CTR by nature.


### The rule, in plain words

A content item deserves a look if it already has real search visibility. Among visible items, the strongest, cleanest lever in the data is the CTR-vs-position gap (Signal 2, confirmed) — pages ranking well but pulling far less click-through than peers at the same position are under-monetizing traffic they've already earned. Staleness (Signal 1, mixed) only tips the action when it's stacked *on top of* a real CTR gap, since staleness alone barely moves the needle here.

**Reason codes (exactly one per row):**

| Reason code | Meaning |
|---|---|
| `low_visibility` | Fewer than 100 impressions in the window — too little traffic to judge |
| `no_position_data` | Has visibility but no measurable average position |
| `stale_ctr_underperformer` | Real CTR gap for its tier **and** not updated in 180+ days |
| `ctr_underperformer` | Real CTR gap for its tier, update recency not (yet) the deciding factor |
| `on_par_or_above` | Visible, has position data, CTR is at or above its tier's norm |


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


In [4]:
def percentile_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")

has_visibility = df["impressions_90d"] >= 100
has_position   = df["avg_position"] > 0
scoreable      = has_visibility & has_position   # only these get a real score

work = df.copy()
work["visibility_score"] = percentile_rank(np.log1p(work["impressions_90d"]))
work["staleness_score"]  = percentile_rank(work["days_since_last_update"])

# CTR gap, normalized against THIS PAGE'S OWN position tier (not a flat threshold)
tier_median_ctr = work.loc[scoreable].groupby("position_tier")["ctr"].transform("median")
work["tier_median_ctr"] = np.nan
work.loc[scoreable, "tier_median_ctr"] = tier_median_ctr
work["ctr_gap_raw"] = (work["tier_median_ctr"] - work["ctr"]).clip(lower=0)
work["ctr_gap_score"] = 0.0
work.loc[scoreable, "ctr_gap_score"] = percentile_rank(work.loc[scoreable, "ctr_gap_raw"])

# readable score: visibility opens the door, ctr gap is the main driver, staleness is a light nudge
work["baseline_action_score"] = 0.0
work.loc[scoreable, "baseline_action_score"] = (
    0.45 * work.loc[scoreable, "visibility_score"]
    + 0.40 * work.loc[scoreable, "ctr_gap_score"]
    + 0.15 * work.loc[scoreable, "staleness_score"]
).clip(0, 1)

def reason_and_action(row):
    if row["impressions_90d"] < 100:
        return "low_visibility", "deprioritize"
    if row["avg_position"] <= 0:
        return "no_position_data", "monitor"
    stale = row["days_since_last_update"] >= 180
    ctr_gap_big = row["ctr_gap_score"] >= 0.75   # top quartile of tier-normalized CTR gap
    if ctr_gap_big and stale:
        return "stale_ctr_underperformer", "refresh_and_fix_ctr"
    if ctr_gap_big:
        return "ctr_underperformer", "ctr_review"
    return "on_par_or_above", "monitor"

reason_action = work.apply(reason_and_action, axis=1, result_type="expand")
work["reason_code"] = reason_action[0]
work["suggested_action"] = reason_action[1]
work["rank"] = work["baseline_action_score"].rank(method="first", ascending=False).astype(int)

out_cols = [
    "rank", "content_id", "client_id", "baseline_action_score", "reason_code", "suggested_action",
    "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier",
    "days_since_last_update", "freshness_tier", "word_count", "content_type", "main_intent",
    "trend_direction",
]
out = work[out_cols].sort_values("rank").reset_index(drop=True)

import os
os.makedirs("../outputs", exist_ok=True)
out.to_csv("../outputs/baseline_action_score.csv", index=False)

print(f"wrote {len(out):,} rows to work/outputs/baseline_action_score.csv\n")
print("action mix:")
print(out["suggested_action"].value_counts())
print("\nreason code mix:")
print(out["reason_code"].value_counts())


wrote 30,000 rows to work/outputs/baseline_action_score.csv

action mix:
suggested_action
monitor                16581
deprioritize            7994
ctr_review              5417
refresh_and_fix_ctr        8
Name: count, dtype: int64

reason code mix:
reason_code
on_par_or_above             16581
low_visibility               7994
ctr_underperformer           5417
stale_ctr_underperformer        8
Name: count, dtype: int64


## 3. Top-10 review

*For each of my top ten: the action, why it's there, and what would make it wrong.*

(The card asks for a top-ten review; the skeleton heading below says "top-20" — going with the card's top ten.)


In [5]:
top10 = out.head(10)[["rank","content_id","suggested_action","reason_code",
                       "impressions_90d","clicks_90d","ctr","avg_position","position_tier",
                       "days_since_last_update","trend_direction"]]
top10


,rank,content_id,suggested_action,reason_code,impressions_90d,clicks_90d,ctr,avg_position,position_tier,days_since_last_update,trend_direction
0,1,content_c8e9d6ab9013,ctr_review,ctr_underperformer,208678,0,0.00,9.7,page_1,104,down
1,2,content_a5dbb404bdc2,ctr_review,ctr_underperformer,79035,59,0.07,8.7,page_1,106,stable
2,3,content_c1fe78bc4e37,ctr_review,ctr_underperformer,134055,43,0.03,7.5,page_1,104,down
3,4,content_b115f7c74779,ctr_review,ctr_underperformer,123469,37,0.03,8.0,page_1,104,up
4,5,content_d0cc5baa4995,ctr_review,ctr_underperformer,83651,28,0.03,6.6,page_1,104,down
5,6,content_36ff89c8214e,ctr_review,ctr_underperformer,295097,154,0.05,7.3,page_1,104,stable
6,7,content_d07ea098353c,ctr_review,ctr_underperformer,63366,21,0.03,9.4,page_1,104,down
7,8,content_4a6607efcb46,ctr_review,ctr_underperformer,128068,17,0.01,2.2,top_3,104,up
8,9,content_9c8299b55f3c,ctr_review,ctr_underperformer,54783,15,0.03,8.5,page_1,104,stable
9,10,content_896bf2cc27b7,ctr_review,ctr_underperformer,66359,28,0.04,4.9,page_1,104,down


1. **`content_c8e9d6ab9013`** — `ctr_review`. 208,678 impressions, page-1 position (9.7), and **zero** clicks — the largest possible CTR gap in the dataset. *Wrong if:* a SERP feature (featured snippet, People Also Ask) is absorbing the clicks structurally — no title/meta change would fix that.
2. **`content_a5dbb404bdc2`** — `ctr_review`. 79,035 impressions at position 8.7, CTR 0.07% vs. a ~0.23% tier norm. *Wrong if:* the ranking query doesn't match the page's intent (e.g. informational content ranking for a transactional query) — refresh effort should target intent, not just CTR wording.
3. **`content_c1fe78bc4e37`** — `ctr_review`. 134,055 impressions, CTR 0.03%, and already **trending down**. *Wrong if:* the decline reflects real demand loss for the topic — refreshing a fading topic wastes effort a CTR fix can't recover.
4. **`content_b115f7c74779`** — `ctr_review`. 123,469 impressions, CTR 0.03%, but trend is **up**. *Wrong if:* it's a young page still climbing — the CTR gap may close naturally as it earns more trust/position, no intervention needed yet.
5. **`content_d0cc5baa4995`** — `ctr_review`. 83,651 impressions, CTR 0.03%, position 6.6, `word_count` missing. *Wrong if:* missing `word_count` here means the underlying content record is incomplete, not that the CTR read is untrustworthy — worth a manual check before acting.
6. **`content_36ff89c8214e`** — `ctr_review`. The single highest-impression item in the top 10 (295,097) but only 154 clicks. *Wrong if:* the ranking keyword is broad/ambiguous and pulling irrelevant impressions — low CTR would then be expected, not fixable by better titles or meta.
7. **`content_d07ea098353c`** — `ctr_review`. Position 9.4 sits right at the `page_1` tier boundary. *Wrong if:* the tier assignment is noisy this close to the edge — the "should have higher CTR for this tier" comparison is less trustworthy near boundaries.
8. **`content_4a6607efcb46`** — `ctr_review`. Position **2.2** (top_3 tier!) with CTR only 0.01% — the cleanest possible mismatch: excellent rank, almost no clicks. *Wrong if:* a competitor's branded result or a snippet is winning the click at that exact query — no content fix reaches that.
9. **`content_9c8299b55f3c`** — `ctr_review`. 54,783 impressions, CTR 0.03%, position 8.5. Same pattern as #2/#5. *Wrong if:* intent mismatch again — worth a manual SERP check before assuming it's fixable.
10. **`content_896bf2cc27b7`** — `ctr_review`. Position 4.9 (strong) but CTR 0.04% and trending down. *Wrong if:* the decline is topic-level demand decay rather than a CTR problem — chasing CTR on a shrinking topic won't move the needle.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


In [6]:
# Weak point: every single top-10 row shares the same reason_code / action.
# The 0.45 visibility weight means the biggest-traffic pages dominate the top of the
# queue regardless of which reason code applies — so `stale_ctr_underperformer` never
# reaches the top 10, even though it exists in the data.
print("full-dataset reason code counts:")
print(out["reason_code"].value_counts())
print()
print("reason codes present in the top 10:", top10["reason_code"].unique().tolist())
print()
print("stale_ctr_underperformer rows, ranked position (min/max):")
stale_rows = out[out["reason_code"] == "stale_ctr_underperformer"]
print(stale_rows[["rank","content_id"]])


full-dataset reason code counts:
reason_code
on_par_or_above             16581
low_visibility               7994
ctr_underperformer           5417
stale_ctr_underperformer        8
Name: count, dtype: int64

reason codes present in the top 10: ['ctr_underperformer']

stale_ctr_underperformer rows, ranked position (min/max):
      rank            content_id
2701  2702  content_fd16e3475c29
3468  3469  content_ea41fe5cf292
3903  3904  content_958a46db26bd
4132  4133  content_02b0d6e30129
4416  4417  content_f488400fca67
5060  5061  content_4f241bad48a3
5200  5201  content_ab27c30d81f4
7783  7784  content_b6e4581523ed


**Weak pick verdict:** the queue is visibility-dominated. Every top-10 pick carries the same `ctr_underperformer` reason code — real signal, but it means the rule never surfaces its more urgent `stale_ctr_underperformer` cases (only 8 exist, and the highest-ranked one sits well outside the top 10) unless I go looking. A next-pass fix would be a small floor or bonus that guarantees at least one or two `stale_ctr_underperformer` rows appear near the top, so staleness isn't drowned out by raw traffic volume.

**Leakage check:**
- The score uses only `impressions_90d`, `avg_position`, `ctr`, `position_tier` (derived from `avg_position`), and `days_since_last_update` — all measured over the *same* trailing 90-day window, no future window involved.
- `trend_direction` and `trend_pct` were **not** used as score inputs anywhere in Section 2 — they're carried into the output CSV only as context columns for the human reviewer, exactly as the data dictionary requires (they're the label source, never a feature).
- No client names, URLs, or private queries appear anywhere in the notebook or the CSV — only pseudonymous `content_id` / `client_id`.


In [7]:
assert "trend_direction" not in ["visibility_score","staleness_score","ctr_gap_score","baseline_action_score"]
score_inputs = {"impressions_90d","avg_position","ctr","position_tier","days_since_last_update"}
label_source = {"trend_direction","trend_pct"}
print("score inputs used:", score_inputs)
print("label-source columns kept OUT of scoring:", label_source, "-> confirmed not used above.")


score inputs used: {'avg_position', 'impressions_90d', 'days_since_last_update', 'ctr', 'position_tier'}
label-source columns kept OUT of scoring: {'trend_pct', 'trend_direction'} -> confirmed not used above.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
